# 02 — การจับคู่ข้อมูล และข้อมูลที่หายไป

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nattaponm/Teach_AOD_PM_PCD/blob/main/02_Matching_and_Missing_Data.ipynb)

## จุดประสงค์การเรียนรู้

เมื่อจบ Notebook นี้ นิสิตสามารถ
1. อธิบายหลักการจับคู่ข้อมูลดาวเทียมกับสถานีทั้งเชิงพื้นที่และเชิงเวลา (collocation)
2. คำนวณ **อัตราการได้ข้อมูล** ของ AOD รายสถานี และแสดงเป็นแผนที่
3. อธิบายได้ว่าเมฆและควันหนาทำให้ข้อมูลหายอย่างไร และทำให้ผลเอนเอียง (bias) อย่างไร
4. สร้างชุด "คู่ข้อมูล" AOD–PM2.5 ที่ผ่านเกณฑ์คุณภาพ เพื่อใช้ใน Notebook 03

**เวลาโดยประมาณ** 90 นาที

> **วิธีเรียน** — อ่านคำอธิบาย → รันทีละ cell → ดูผล → ตอบคำถาม "ลองคิด" ในใจก่อนรัน cell ถัดไป
> ไม่แนะนำให้กด *Run all* ในรอบแรก

## 1. หลักการ: จับคู่ให้ "ที่เดียวกัน เวลาเดียวกัน"

### เชิงพื้นที่
สถานีเป็น **จุด** ส่วน pixel ดาวเทียมกว้าง 1 km และค่าของ pixel เดียวมี noise สูง
Notebook 00 จึงใช้ค่าเฉลี่ยของ pixel ในวงกลมรัศมี 1.5 km รอบสถานี (เส้นผ่านศูนย์กลาง 3 km)
และบันทึกจำนวน pixel ที่มีค่าไว้ในคอลัมน์ `aod_n_pixels` ซึ่งนับทุก pixel ที่วงกลมแตะ จึงมีได้สูงสุดราว 14 pixel

### เชิงเวลา
```
AOD    : ค่าขณะดาวเทียมผ่าน  ~10:30 น. (Terra) และ ~13:30 น. (Aqua)
PM2.5  : ค่าเฉลี่ย 24 ชั่วโมงของวันเดียวกัน
```
เราจับคู่ด้วย **วันที่เดียวกัน** ซึ่งเป็นวิธีมาตรฐาน แต่ต้องจำไว้ว่าเวลาไม่ตรงกันสนิท

### สมการสำคัญ: อัตราการได้ข้อมูล (data availability)

$$A = \frac{N_{\text{valid}}}{N_{\text{total}}} \times 100\%$$

| สัญลักษณ์ | ความหมาย |
|---|---|
| $N_{\text{valid}}$ | จำนวนวันที่ดาวเทียมให้ค่า AOD คุณภาพดี |
| $N_{\text{total}}$ | จำนวนวันทั้งหมดในช่วงที่ศึกษา |

**ตัวอย่าง** ช่วงศึกษามี 89 วัน สถานีหนึ่งได้ค่า AOD 40 วัน → $A = 40/89 \times 100 = 44.9\%$

## 0. เตรียมเครื่องมือ

Cell ชุดนี้เหมือนกันทุก Notebook ทำ 3 อย่าง คือ โหลด library, ดาวน์โหลดข้อมูลจาก GitHub ของรายวิชา และสร้างฟังก์ชันแผนที่มาตรฐาน
รันทีละ cell ตามลำดับ ไม่จำเป็นต้องอ่านโค้ดส่วนนี้ละเอียดในรอบแรก

In [ ]:
from pathlib import Path
import urllib.request
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

warnings.filterwarnings("ignore")

YEAR = 2023
DATA_URL = "https://raw.githubusercontent.com/nattaponm/Teach_AOD_PM_PCD/main/data/"
DATA_DIR = Path("data"); DATA_DIR.mkdir(exist_ok=True)
FIG_DIR = Path("figures"); FIG_DIR.mkdir(exist_ok=True)

def fetch(name):
    # ดาวน์โหลดไฟล์จาก GitHub มาเก็บในโฟลเดอร์ data (โหลดครั้งเดียว)
    path = DATA_DIR / name
    if not path.exists():
        urllib.request.urlretrieve(DATA_URL + name, path)
    return path

In [ ]:
# รูปแบบภาพสำหรับงานตีพิมพ์: ตัวอักษรอ่านง่าย เส้นบาง ความละเอียด 300 dpi
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "legend.fontsize": 8, "legend.frameon": False,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
})

UNIT = "µg m$^{-3}$"

def save_fig(fig, name):
    # บันทึกทั้ง PNG (ใส่รายงาน) และ PDF (ส่งวารสาร)
    fig.savefig(FIG_DIR / f"{name}.png")
    fig.savefig(FIG_DIR / f"{name}.pdf")
    print("บันทึกแล้ว:", FIG_DIR / f"{name}.png", "และ .pdf")

In [ ]:
# ขอบเขต 77 จังหวัด และเส้นขอบประเทศ (รวมทุกจังหวัดเป็นรูปเดียว)
provinces = gpd.read_file(fetch("provinces.geojson"))
country = gpd.GeoSeries([provinces.buffer(0.002).union_all().buffer(-0.002)], crs=provinces.crs)

MAP_EXTENT = [97.0, 106.0, 5.3, 20.8]   # west, east, south, north

def base_map(ax, title=None, province_color="0.45"):
    # วาดองค์ประกอบแผนที่มาตรฐาน: จังหวัด ประเทศ พิกัด มาตราส่วน ทิศเหนือ
    provinces.boundary.plot(ax=ax, linewidth=0.25, color=province_color, zorder=3)
    country.boundary.plot(ax=ax, linewidth=0.9, color="black", zorder=4)

    ax.set_xlim(MAP_EXTENT[0], MAP_EXTENT[1]); ax.set_ylim(MAP_EXTENT[2], MAP_EXTENT[3])
    ax.set_aspect(1 / np.cos(np.deg2rad(13)))          # แก้การยืดของลองจิจูดที่ละติจูด ~13°N
    ax.set_xticks(np.arange(98, 107, 2)); ax.set_yticks(np.arange(6, 21, 2))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}°E"))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}°N"))
    ax.tick_params(direction="in", top=True, right=True, length=3)
    ax.grid(linewidth=0.3, color="0.8", linestyle=":", zorder=0)

    # มาตราส่วน 200 km (1 องศาลองจิจูด = 111.32 x cos(ละติจูด) km)
    lat0, lon0 = 6.1, 102.9
    width = 200 / (111.32 * np.cos(np.deg2rad(lat0)))
    ax.plot([lon0, lon0 + width], [lat0, lat0], color="black", linewidth=1.5, zorder=5)
    ax.text(lon0 + width / 2, lat0 + 0.18, "200 km", ha="center", va="bottom", fontsize=7)

    # ลูกศรทิศเหนือ
    ax.annotate("N", xy=(105.2, 20.2), xytext=(105.2, 19.2), ha="center", va="center", fontsize=9,
                arrowprops=dict(arrowstyle="-|>", color="black", linewidth=1))
    ax.set_xlabel(""); ax.set_ylabel("")              # พิกัดบอกอยู่ที่ตัวเลขแกนแล้ว
    if title:
        ax.set_title(title)
    return ax

print("จำนวนจังหวัด:", len(provinces))

## 2. เปิดตารางจับคู่

ไฟล์ `aod_pm_matchup.csv` มีหนึ่งแถวต่อ "หนึ่งสถานี หนึ่งวัน"

In [ ]:
m = pd.read_csv(fetch("aod_pm_matchup.csv"), parse_dates=["date"])
print("จำนวนแถว:", len(m), "| สถานี:", m["station"].nunique(), "| วัน:", m["date"].nunique())
m[["date", "station", "pm25", "aod_055", "aod_n_pixels", "column_wv", "region"]].head()

In [ ]:
# แต่ละคอลัมน์มีค่ากี่เปอร์เซ็นต์
(m[["pm25", "aod_055"]].notna().mean() * 100).round(1)

> 💭 **ลองคิด** — ทำไมสถานีภาคพื้นดินจึงมีข้อมูลเกือบครบ แต่ดาวเทียมมีน้อยกว่ามาก

## 3. ดาวเทียมเห็นกี่วัน

### ภาพที่ 1 — แผนที่อัตราการได้ข้อมูล AOD รายสถานี

In [ ]:
m["has_aod"] = m["aod_055"].notna()

avail = (m.groupby(["station", "lat", "lon", "region"])["has_aod"].mean() * 100).rename("availability").reset_index()

fig, ax = plt.subplots(figsize=(3.6, 6.2))
base_map(ax, f"AOD availability, Feb–Apr {YEAR}")
sc = ax.scatter(avail["lon"], avail["lat"], c=avail["availability"], cmap="viridis",
                vmin=0, vmax=100, s=26, edgecolor="black", linewidth=0.3, zorder=6)
fig.colorbar(sc, ax=ax, shrink=0.55, pad=0.03, label="Days with valid AOD (%)")
save_fig(fig, "fig04_aod_availability_map")
plt.show()

avail.groupby("region")["availability"].mean().round(1)

### ภาพที่ 2 — วันใดมีข้อมูล วันใดไม่มี

แต่ละแถวคือหนึ่งสถานี (เรียงจากใต้ขึ้นเหนือ) แต่ละคอลัมน์คือหนึ่งวัน ช่องสีขาวคือวันที่ไม่มีค่า AOD

In [ ]:
order = avail.sort_values("lat")["station"]
grid = m.pivot(index="station", columns="date", values="aod_055").loc[order]

fig, ax = plt.subplots(figsize=(7.0, 3.6))
im = ax.imshow(np.ma.masked_invalid(grid.values), aspect="auto", cmap="YlOrBr", vmin=0, vmax=1.5,
               interpolation="nearest")
month_starts = [i for i, d in enumerate(grid.columns) if d.day == 1]
ax.set_xticks(month_starts)
ax.set_xticklabels([grid.columns[i].strftime("%d %b") for i in month_starts])
ax.set_yticks([0, len(grid) - 1]); ax.set_yticklabels(["South", "North"])
ax.set_xlabel(f"Date ({YEAR})"); ax.set_ylabel("Stations sorted by latitude")
fig.colorbar(im, ax=ax, pad=0.02, extend="max", label="AOD (white = no retrieval)")
save_fig(fig, "fig05_aod_station_day_heatmap")
plt.show()

> 💭 **ลองคิด** — ช่องขาวเรียงเป็น "แถบแนวตั้ง" (หลายสถานีหายพร้อมกันในวันเดียว) หรือกระจายสุ่ม รูปแบบนี้บอกสาเหตุอะไร

## 4. ข้อมูลที่หายทำให้ผลเอนเอียงหรือไม่

ถ้าดาวเทียมมองไม่เห็นในวันที่ฝุ่นหนาที่สุด (เพราะอัลกอริทึมจัดควันหนาเป็นเมฆ) หรือในวันที่ฝนตกอากาศสะอาด
ค่าเฉลี่ยที่คิดจาก "วันที่มี AOD" ก็จะไม่ใช่ตัวแทนของทุกวัน เรียกว่า **sampling bias**

ทดสอบได้โดยเทียบ PM2.5 เฉลี่ยของทุกวัน กับเฉพาะวันที่มี AOD

### ภาพที่ 3 — PM2.5 เฉลี่ย: ทุกวัน เทียบกับ เฉพาะวันที่ดาวเทียมเห็น

In [ ]:
has_pm = m.dropna(subset=["pm25"])
compare = pd.DataFrame({
    "All days": has_pm.groupby("region")["pm25"].mean(),
    "Days with AOD": has_pm[has_pm["has_aod"]].groupby("region")["pm25"].mean(),
})
compare["difference (%)"] = (compare["Days with AOD"] / compare["All days"] - 1) * 100

x = np.arange(len(compare)); w = 0.38
fig, ax = plt.subplots(figsize=(5.0, 3.0))
ax.bar(x - w / 2, compare["All days"], w, color="0.55", label="All days")
ax.bar(x + w / 2, compare["Days with AOD"], w, color="#d95f02", label="Days with valid AOD")
ax.set_xticks(x); ax.set_xticklabels(compare.index)
ax.set_ylabel(f"Mean PM$_{{2.5}}$ ({UNIT})")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.0), ncol=2)
ax.spines[["top", "right"]].set_visible(False)
save_fig(fig, "fig06_sampling_bias")
plt.show()

compare.round(1)

## 5. สร้างชุดคู่ข้อมูลที่ผ่านเกณฑ์คุณภาพ

ค่า AOD ที่มาจาก pixel เดียวไม่น่าเชื่อถือเท่าค่าที่เฉลี่ยจากหลาย pixel
เกณฑ์ที่ใช้ในชุดบทเรียนนี้คือ **ต้องมีอย่างน้อย 3 pixel** ในวงรอบสถานี

In [ ]:
MIN_PIXELS = 3

pairs = m.dropna(subset=["pm25", "aod_055"])
pairs = pairs[pairs["aod_n_pixels"] >= MIN_PIXELS]

print("แถวทั้งหมด            :", len(m))
print("มีทั้ง PM2.5 และ AOD   :", len(m.dropna(subset=["pm25", "aod_055"])))
print(f"ผ่านเกณฑ์ >= {MIN_PIXELS} pixel   :", len(pairs))

## 6. สรุป

```
ไม่มีค่า AOD            ≠  อากาศสะอาด
ค่าเฉลี่ยของวันที่ฟ้าเปิด   ≠  ค่าเฉลี่ยของทุกวัน
จำนวนคู่ข้อมูลมาก        ≠  คุณภาพคู่ข้อมูลดี
```

เกณฑ์คุณภาพเป็นการ **แลกเปลี่ยน**: เข้มขึ้นได้ข้อมูลน่าเชื่อถือขึ้นแต่จำนวนน้อยลง
Notebook ถัดไปจะใช้ชุด `pairs` นี้หาความสัมพันธ์

---
## แบบฝึกหัด

### ข้อ 1 (ฝึกทำ) — อัตราการได้ข้อมูลของสถานีที่เลือก
เลือกสถานีหนึ่งแห่งจากตาราง `avail` แล้วคำนวณอัตราการได้ข้อมูลด้วยสมการ $A$ โดยนับ $N_{\text{valid}}$ และ $N_{\text{total}}$ เอง
ตรวจว่าตรงกับค่าในตาราง `avail`

In [ ]:
# เขียนโค้ดของคุณที่นี่
my_station = avail["station"].iloc[0]     # เปลี่ยนเป็นรหัสสถานีที่สนใจ เช่น "36T"

my_availability = None   # ใส่คำตอบเป็นร้อยละ

In [ ]:
#@title ตรวจคำตอบข้อ 1 (ลองทำเองก่อน แล้วดับเบิลคลิกเพื่อดูโค้ด)
_expected = avail.set_index("station").loc[my_station, "availability"]
if my_availability is None:
    print("ยังไม่ได้ใส่คำตอบใน my_availability")
elif abs(my_availability - _expected) < 0.1:
    print("ถูกต้อง")
else:
    print("ยังไม่ใช่ ตรวจว่าตัวหารคือจำนวนวันทั้งหมดของสถานีนั้น ไม่ใช่จำนวนแถวทั้งตาราง")

In [ ]:
#@title เฉลยข้อ 1 (ลองทำเองก่อน แล้วดับเบิลคลิกเพื่อดูโค้ด)
one = m[m["station"] == my_station]
n_valid = one["aod_055"].notna().sum()
n_total = len(one)
print(f"สถานี {my_station}: N_valid = {n_valid}, N_total = {n_total}, A = {n_valid / n_total * 100:.1f}%")

### ข้อ 2 (ฝึกคิด) — เดือนใดดาวเทียมเห็นมากที่สุด
คำนวณอัตราการได้ข้อมูลเฉลี่ยทั้งประเทศแยกรายเดือน (ก.พ. มี.ค. เม.ย.)

In [ ]:
# เขียนโค้ดของคุณที่นี่
# แนวทาง: m.groupby(m["date"].dt.month)["has_aod"].mean() * 100

In [ ]:
#@title เฉลยข้อ 2 (ลองทำเองก่อน แล้วดับเบิลคลิกเพื่อดูโค้ด)
(m.groupby(m["date"].dt.month)["has_aod"].mean() * 100).round(1).rename("availability (%)").rename_axis("month")

<details>
<summary><b>คำอธิบายข้อ 2</b> (คลิกเพื่อเปิด)</summary>

`dt.month` ดึงเลขเดือนจากวันที่ จึงใช้เป็นกลุ่มใน `groupby` ได้โดยไม่ต้องสร้างคอลัมน์ใหม่

ในการตีความ ให้นึกถึงสองปัจจัยที่ทำให้ค่าลดลง คือ เมฆ (เพิ่มขึ้นเมื่อเข้าใกล้ฤดูฝนและช่วงพายุฤดูร้อนในเดือนเมษายน)
และควันหนาที่อัลกอริทึมจัดเป็นเมฆ (มากในช่วงไฟป่ารุนแรง)

</details>

### ข้อ 3 (ฝึกคิด) — เกณฑ์เข้มขึ้น ข้อมูลเหลือเท่าใด
นับจำนวนคู่ข้อมูลที่เหลือเมื่อใช้เกณฑ์ `MIN_PIXELS` เท่ากับ 1, 3, 5 และ 7

In [ ]:
# เขียนโค้ดของคุณที่นี่
# แนวทาง: วนลูป for k in [1, 3, 5, 7]: แล้วนับแถวที่ aod_n_pixels >= k

In [ ]:
#@title เฉลยข้อ 3 (ลองทำเองก่อน แล้วดับเบิลคลิกเพื่อดูโค้ด)
both = m.dropna(subset=["pm25", "aod_055"])
for k in [1, 3, 5, 7]:
    n = (both["aod_n_pixels"] >= k).sum()
    print(f"อย่างน้อย {k} pixel: {n:5d} คู่ ({n / len(both) * 100:5.1f}% ของคู่ทั้งหมด)")

<details>
<summary><b>คำอธิบายข้อ 3</b> (คลิกเพื่อเปิด)</summary>

ไม่มีเกณฑ์ใด "ถูก" เพียงค่าเดียว งานวิจัยต้อง **รายงานเกณฑ์ที่ใช้** และควรทดสอบว่าผลสรุปเปลี่ยนหรือไม่เมื่อเปลี่ยนเกณฑ์
(เรียกว่า sensitivity analysis) ซึ่งนิสิตจะได้ลองใน Notebook 03

</details>

### ข้อ 4 (ฝึกตีความ)
ดูตาราง `compare` ในหัวข้อ 4 ภาคใดที่ "วันที่ดาวเทียมเห็น" มี PM2.5 เฉลี่ยต่างจาก "ทุกวัน" มากที่สุด
ถ้านำ AOD ไปประเมินการรับสัมผัสฝุ่นของประชากรในภาคนั้น ผลจะคลาดเคลื่อนไปทางใด

<details>
<summary><b>แนวคำตอบข้อ 4</b> (คลิกเพื่อเปิด)</summary>

อ่านจากคอลัมน์ `difference (%)`
- ค่า **ติดลบ** แปลว่าดาวเทียมพลาดวันที่ฝุ่นสูง (เช่น ควันหนาถูกจัดเป็นเมฆ) การประเมินจาก AOD จะ **ต่ำกว่าความจริง**
- ค่า **บวก** แปลว่าดาวเทียมพลาดวันที่อากาศสะอาด (เช่น วันฝนตกมีเมฆ) การประเมินจะ **สูงกว่าความจริง**

ทั้งสองกรณีไม่ได้แก้ด้วยการมีข้อมูลมากขึ้น แต่ต้องยอมรับและรายงานเป็นข้อจำกัด หรือใช้วิธีเติมช่องว่าง (gap-filling) ซึ่งเป็นหัวข้อวิจัยขั้นสูง

</details>

### ต่อยอดสำหรับ ป.โท
ออกแบบการทดลองเพื่อตอบว่า ขนาดหน้าต่างเชิงพื้นที่ (1 × 1, 3 × 3, 5 × 5 km) มีผลต่อความสัมพันธ์ AOD–PM2.5 อย่างไร
ต้องแก้ส่วนใดของ Notebook 00 และควรคุมตัวแปรใดให้คงที่

*แนวทาง* — แก้ค่า `BUFFER_M` แล้วรันส่วน C ใหม่สำหรับแต่ละขนาด เปรียบเทียบบน **ชุดสถานี-วันเดียวกัน** เท่านั้น
มิฉะนั้นจะแยกไม่ออกว่าผลต่างมาจากขนาดหน้าต่างหรือจากจำนวนข้อมูลที่ไม่เท่ากัน